# MuonClip angular/radial RG diagnostics

This exploratory notebook now uses the **same canonical angular power-law tail test** as the strict notebook in `notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb`.

The important rule is identical in both notebooks:

```python
powerlaw.Fit(all_positive_projective_angular_values,
             discrete=False, verbose=False)
```

No manual `xmin` is supplied. No `xmax` is supplied. The package searches for the tail start by its MLE/KS procedure and then fits the **largest values** `x >= xmin` all the way through the largest observed value.

The notebook runs both the broader angular/radial baseline pipeline and the dedicated far-tail/null pipeline. Use `RG_MATRIX_NAME` to select a layer when inspecting the returned tables; the analysis itself still evaluates all six matrices so the fit contract is identical everywhere.


In [ ]:
from pathlib import Path
import os
import sys

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch Jupyter from the rg_optimizers repository"
    )

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)


In [ ]:
from IPython.display import display
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_weightwatcher_pipeline import run_analysis
from rg_nanogpt_one_head.angular_powerlaw_tail import run_powerlaw_tail_analysis

CONFIG = AnalysisConfig.from_env()
MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")
print(CONFIG)
print("RG_MATRIX_NAME =", MATRIX_NAME)


In [ ]:
# Broader initial/final angular + radial diagnostics.
RESULTS, RESOLVED_RUN, ANALYSIS_MANIFEST = run_analysis(CONFIG)
display(RESULTS)


In [ ]:
# Far-tail power-law test against matched random-angular nulls.
TAIL_RESULTS, _ = run_powerlaw_tail_analysis(CONFIG)

selected = TAIL_RESULTS[TAIL_RESULTS["matrix_name"] == MATRIX_NAME]
print("RUN_DIR =", RESOLVED_RUN.run_dir)
print("INITIAL =", RESOLVED_RUN.initial_path)
print("FINAL =", RESOLVED_RUN.final_path)
print("OUTPUT =", RESOLVED_RUN.output_dir)
display(selected if len(selected) else TAIL_RESULTS)


## What to inspect

For the selected matrix inspect both `tilt` and `twist` (for square attention matrices, tilt is structurally trivial and twist is the useful sector).

The key files are:

```text
<MATRIX>_<tilt|twist>_powerlaw_pdf_loglog.png
<MATRIX>_<tilt|twist>_powerlaw_pdf_linear.png
<MATRIX>_<tilt|twist>_powerlaw_cdf.png
<MATRIX>_<tilt|twist>_powerlaw_ccdf_loglog.png
<MATRIX>_<tilt|twist>_far_tail_zoom_ccdf.png
angular_powerlaw_far_tail_summary.csv
```

The far-tail zoom overlays the trained CCDF, the random-angular median and 95% envelope, the package-selected trained `xmin`, and the null median `xmin`. The zoom is display-only: it does **not** change which values are passed to `powerlaw.Fit`.
